# CyTOF Data Loading and Preparation

This notebook shows how to load raw CyTOF data exported from OMIQ and prepare a clean,  
annotated cell matrix ready for correction.

**Workflow:**
1. Load the OMIQ metadata file (sample sheet)
2. Format it into the standard metadata table expected by the pipeline
3. Configure the pipeline (`CyTOFContext`)
4. Call `load_data_from_omiq` to get the annotated cell matrix
5. Inspect and save the matrix for use in `Correction.ipynb`

The output of this notebook is a single CSV file containing all cells, annotated  
with `sample_name`, `plate`, `dataset`, `panel`, `condition`, and all marker columns.

In [1]:
import sys
import os
import json
import pandas as pd

# Add the project root to the Python path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from src import load_data_from_omiq, get_config, CyTOFContext

/Users/eloiseberson/Documents/Code/CyTOF/CyTOF-Prep/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


If you installed Somoclu with pip on macOS, this typically means missing a linked library. If you compiled Somoclu with GCC, please make sure you have set DYLD_LIBRARY_PATH to include the GCC path. For more information, please refer to the documentation.


## Step 1 — Load the OMIQ metadata file

OMIQ exports a sample sheet (Excel or CSV) that lists the filenames, sample labels,  
and any experimental annotations. Load it here and display its structure.

In [2]:
# Path to the OMIQ-exported metadata file (adjust sheet_name if needed)
omiq_metadata_path = "../data/cohort_2/OMIQ_metadata-OHSU 18 patient dataset.xlsx"
df_omiq = pd.read_excel(omiq_metadata_path, sheet_name=2)

print(f"Shape: {df_omiq.shape}")
display(df_omiq.head())

Shape: (111, 6)


,OmiqID,Filename,Subject,Timepoint,Ex-vivo treatment,Notes
0,F6563907696066560,Plate6_P15 Dx unknown4.fcs,P15,Enrollment,unknown,labelled lost
1,F4734320347447296,Plate6_P15 Dx unknown3.fcs,P15,Enrollment,unknown,labelled lost
2,F5860220254289920,Plate6_P15 Dx unknown2.fcs,P15,Enrollment,unknown,labelled lost
3,F5297270300868608,Plate6_P15 Dx unknown1.fcs,P15,Enrollment,unknown,labelled lost
4,F6423170207711232,Plate6_P15 C1D8 Untreated.fcs,P15,C1 Day 8,none,NaN


## Step 2 — Format into the standard metadata table

The pipeline expects a DataFrame with these columns:

| Column | Description |
|---|---|
| `file_name` | Filename as stored in `data_dir` (e.g. `Plate1_Sample01.csv`) |
| `sample_name` | Biological sample identifier |
| `plate` | Plate/batch label (must match `ref_plate` in config) |
| `dataset` | Dataset/timepoint identifier (e.g. `D0`) |
| `panel` | Panel name (e.g. `Panel1`) |
| `condition` | Experimental condition (optional but recommended) |

Below is an example of how to derive `plate` and `sample_name` from the OMIQ filename  
when the naming convention is `<Plate>_<SampleName>.csv`.

In [4]:
my_metadata = pd.DataFrame()

# file_name: convert .fcs → .csv to match exported files
my_metadata['file_name'] = df_omiq['Filename'].str.replace('.fcs', '.csv', regex=False)

# plate: first token before '_' when the filename contains 'Plate'
def parse_plate(filename):
    parts = filename.split('_')
    return parts[0] if 'Plate' in parts[0] else 'Plate1'

my_metadata['plate'] = my_metadata['file_name'].apply(parse_plate)

# sample_name: strip plate prefix and file extension
my_metadata['sample_name'] = my_metadata.apply(
    lambda row: row['file_name']
        .replace(row['plate'] + '_', '')
        .replace('.csv', ''),
    axis=1
)

# Fixed columns for this cohort
my_metadata['dataset'] = "OHSU_18"
my_metadata['panel']   = "Panel1"

# Condition from OMIQ sheet (adjust column name as needed)
my_metadata['condition'] = my_metadata['sample_name'].str.split(" ", n=1).str[1]

print("Formatted metadata:")
display(my_metadata.head(10))
print(f"\nPlates found: {sorted(my_metadata['plate'].unique())}")
print(f"Samples found: {sorted(my_metadata['sample_name'].unique())}")

Formatted metadata:


,file_name,plate,sample_name,dataset,panel,condition
0,Plate6_P15 Dx unknown4.csv,Plate6,P15 Dx unknown4,OHSU_18,Panel1,Dx unknown4
1,Plate6_P15 Dx unknown3.csv,Plate6,P15 Dx unknown3,OHSU_18,Panel1,Dx unknown3
2,Plate6_P15 Dx unknown2.csv,Plate6,P15 Dx unknown2,OHSU_18,Panel1,Dx unknown2
3,Plate6_P15 Dx unknown1.csv,Plate6,P15 Dx unknown1,OHSU_18,Panel1,Dx unknown1
4,Plate6_P15 C1D8 Untreated.csv,Plate6,P15 C1D8 Untreated,OHSU_18,Panel1,C1D8 Untreated
5,Plate6_P6_1 EOT Untreated.csv,Plate6,P6_1 EOT Untreated,OHSU_18,Panel1,EOT Untreated
6,Plate6_P6_1 Dx Ven.csv,Plate6,P6_1 Dx Ven,OHSU_18,Panel1,Dx Ven
7,Plate6_P6_1 Dx Untreated.csv,Plate6,P6_1 Dx Untreated,OHSU_18,Panel1,Dx Untreated
8,Plate6_P6_1 Dx Rux.csv,Plate6,P6_1 Dx Rux,OHSU_18,Panel1,Dx Rux
9,Plate6_P6_1 Dx Combo.csv,Plate6,P6_1 Dx Combo,OHSU_18,Panel1,Dx Combo



Plates found: ['Plate1', 'Plate2', 'Plate3', 'Plate4', 'Plate5', 'Plate6']
Samples found: ['2100791 C1D8 Untreated', '2100826 C1D8 Untreated', 'Healthy BM 28007', 'Healthy BM 36895', 'Healthy BM 38624', 'Healthy BM 5871 Basal', 'Healthy BM 5871 IL3', 'Healthy BM 5871 IL6', 'P1 C1D8 Untreated', 'P1 Dx Combo', 'P1 Dx Rux', 'P1 Dx Untreated', 'P1 Dx Ven', 'P10 C1D8 Untreated', 'P10 Dx Combo', 'P10 Dx Rux', 'P10 Dx Untreated', 'P10 Dx Ven', 'P11 C2D8 Untreated', 'P11 Dx Combo', 'P11 Dx Rux', 'P11 Dx Untreated', 'P11 Dx Ven', 'P12 C1D15 Untreated', 'P12 C1D8 Untreated', 'P12 Dx Combo', 'P12 Dx Rux', 'P12 Dx Untreated', 'P12 Dx Ven', 'P13 Dx Combo', 'P13 Dx Rux', 'P13 Dx Untreated', 'P13 Dx Ven', 'P14 C1D15 Untreated', 'P14 C1D8 Untreated', 'P14 Dx Combo', 'P14 Dx Rux', 'P14 Dx Untreated', 'P14 Dx Ven', 'P15 C1D8 Untreated', 'P15 Dx unknown1', 'P15 Dx unknown2', 'P15 Dx unknown3', 'P15 Dx unknown4', 'P16 C1D15 Untreated', 'P16 Dx Combo', 'P16 Dx Rux', 'P16 Dx Untreated', 'P16 Dx Ven', 'P17 

In [5]:
my_metadata["condition"].unique()

array(['Dx unknown4', 'Dx unknown3', 'Dx unknown2', 'Dx unknown1',
       'C1D8 Untreated', 'EOT Untreated', 'Dx Ven', 'Dx Untreated',
       'Dx Rux', 'Dx Combo', 'BM 38624', 'C1D15 Untreated', 'BM 5871 IL6',
       'BM 5871 IL3', 'BM 5871 Basal', 'BM 28007', 'C1D1 Untreated',
       'EOT Ven', 'EOT Rux', 'EOT Combo', 'BM 36895', 'C2D8 Untreated',
       'DxmCombo', 'C3D1 Untreated', 'C2D15 Untreated', 'C1D22 Untreated'],
      dtype=object)

## Step 3 — Configure the pipeline

The `CyTOFContext` centralises all pipeline settings.  
Adjust the values below to match your panel design and directory layout.

In [6]:
my_config = {
    # Directory containing the OMIQ-exported CSV/FCS files
    "data_dir": "../data/cohort_2/omiq_exported_data_OHSU_ELOISE/",

    # Where corrected results will be written (only used by run_cytof_correction)
    "output_dir": "../results/Correction/",

    # The plate whose anchor cells define the reference distribution
    "ref_plate": "Plate1",

    # Anchor sample name per dataset — must appear in every plate
    "anchors": {
        "OHSU_18": "Healthy BM 38624"
    },

    # Columns that carry metadata (not biological signal)
    "meta_cols": [
        "original_filename", "Event_length", "Time",
        "dataset", "panel", "plate", "sample_name", "condition",
    ],

    # OmiqFilter gate names used for cell type assignment
    "celltypes": [
        "89Y_CD45+/T-cells", "89Y_CD45+/B-cells",
        # "bulk_leukocytes",
          "megakaryocytic", "myeloid-enriched", "immature-like"
    ],

    # Cell types to apply time-drift correction to
    "celltype2correct": [
        "89Y_CD45+/T-cells", "89Y_CD45+/B-cells",
        # "bulk_leukocytes", 
        "megakaryocytic", "myeloid-enriched","immature-like"
    ],

    # The marker used as a time-drift reference (e.g. DNA intercalator)
    "stable_marker": "DNA",

    # Biological markers to include in batch correction
    "true_markers": [
        'CD45', 'DNA', 'CD14', 'CD33', 'CD56', 'CD34', 'CD61', 'CD94', 'CD38',
       'CD45RA', 'PKM1', 'CD3_CD19', 'PKM2', 'CD117', 'GSS', 'CD64', 'BCLXL',
       'CS', 'MDM2', 'pSTAT5', 'CD123', 'ENO1', 'pSTAT1', 'MCT4', 'CytC',
       'VDAC1', 'ATPA5', 'BCL2', 'HK2', 'p53', 'pSTAT3', 'MCL1', 'Caspase3',
       'ASCT2', 'LDHA', 'pErk', 'pP38', 'CD135', 'CD58', 'pS6', 'CPT1A',
       'CD36', 'MCT1', 'pCreb', 'CD68', 'cParp', 'MPO', 'GLUT1', 'CD11b'
        # Add all your panel markers here
    ],

    # Metal → marker name mapping (for OMIQ channel renaming, leave empty if not needed)
    "metal_to_marker": {},

    # Gating column names in the OMIQ export that define CD45+ populations
    "celltype_gating": [
        "89Y_CD45+/T-cells", "89Y_CD45+/B-cells",#'CD45high',
        # "bulk_leukocytes", 
        "megakaryocytic", "myeloid-enriched", "immature-like"
    ]
}

config = get_config(config_dict=my_config)
ctx    = CyTOFContext(config)
print("CyTOFContext created successfully")
print(f"  data_dir  : {ctx.data_dir}")
print(f"  ref_plate : {ctx.ref_plate}")
print(f"  anchors   : {ctx.anchors}")

CyTOFContext created successfully
  data_dir  : ../data/cohort_2/omiq_exported_data_OHSU_ELOISE/
  ref_plate : Plate1
  anchors   : {'OHSU_18': 'Healthy BM 38624'}


## Step 4 — Load and annotate the cell matrix

`load_data_from_omiq` reads every matching file, harmonizes marker names,  
applies the marker schema, assigns cell types, and annotates each cell with  
the metadata from `my_metadata`.

Use the `dataset` and `panel` arguments to restrict loading to a subset.

In [7]:

# celltype_col: name of the column in your OMIQ CSV that contains cell type labels.
# Open one of your exported CSV files and find the column that holds population names
# (e.g. "Population", "cell_type", "Gate", ...). Set it here.
# If your files use OmiqFilter binary flag columns instead, set celltype_col=None.
celltype_col = "OmiqFilter"   # <-- adjust to match your OMIQ export

matrix, markers = load_data_from_omiq(
    data_dir     = ctx.data_dir,
    metadata_df  = my_metadata,
    ctx          = ctx,
    dataset      = "OHSU_18",   # set to None to load all datasets
    panel        = None,        # set to 'Panel1' etc. to restrict to one panel
    celltype_col = celltype_col,
)

print(f"\nMatrix shape : {matrix.shape}")
print(f"Markers ({len(markers)}): {markers}")


  [CT] column 'OmiqFilter' → values: ['Unfiltered' 'CD45inter' '89Y_CD45+/T-cells' 'myeloid-enriched' 'CD45low']
[LOAD] Plate5_P16 Dx Untreated.csv | n_cells=133519
  [CT] column 'OmiqFilter' → values: ['89Y_CD45+/T-cells' 'CD45inter' '89Y_CD45+/B-cells' 'cells' 'Unfiltered']
[LOAD] Plate2_P10 Dx Untreated.csv | n_cells=10856
  [CT] column 'OmiqFilter' → values: ['CD45high' '89Y_CD45+/T-cells' '89Y_CD45+/B-cells' 'cells' 'CD45inter']
[LOAD] Plate2_P10 Dx Rux.csv | n_cells=12431
  [CT] column 'OmiqFilter' → values: ['CD45high' 'myeloid-enriched' 'megakaryocytic' 'CD45inter'
 '89Y_CD45+/T-cells']
[LOAD] Plate5_P16 Dx Combo.csv | n_cells=185999
  [CT] column 'OmiqFilter' → values: ['89Y_CD45+/T-cells' 'CD45high' 'megakaryocytic' 'Unfiltered' 'cells']
[LOAD] Plate5_Healthy BM 5871 IL6.csv | n_cells=73066
  [CT] column 'OmiqFilter' → values: ['CD45inter' 'megakaryocytic' 'myeloid-enriched' 'immature-like' 'cells']
[LOAD] Plate6_P6_1 EOT Untreated.csv | n_cells=34101
  [CT] column 'OmiqFilte

In [8]:
print(matrix.celltypes.unique())
# mapping_ = {"immature-like": "bulk_leukocytes", "myeloid-enriched": "bulk_leukocytes","CD"}
matrix = matrix[matrix.celltypes.isin(ctx.celltype_gating)]
print(f"\nMatrix shape after cell type filtering: {matrix.shape}")

['Unfiltered' 'CD45inter' '89Y_CD45+/T-cells' 'myeloid-enriched' 'CD45low'
 'CD45high' 'megakaryocytic' 'immature-like' 'cells' '89Y_CD45-'
 '89Y_CD45+' '89Y_CD45+/B-cells' 'CD45low/GLUT1high' 'bulk_leukocytes']

Matrix shape after cell type filtering: (6661281, 59)


In [9]:
matrix.columns

Index(['original_filename', 'original_filename', 'Event_length', 'Time',
       'dataset', 'panel', 'plate', 'sample_name', 'condition', 'celltypes',
       'CD45', 'DNA', 'CD14', 'CD33', 'CD56', 'CD34', 'CD61', 'CD94', 'CD38',
       'CD45RA', 'PKM1', 'CD3_CD19', 'PKM2', 'CD117', 'GSS', 'CD64', 'BCLXL',
       'CS', 'MDM2', 'pSTAT5', 'CD123', 'ENO1', 'pSTAT1', 'MCT4', 'CytC',
       'VDAC1', 'ATPA5', 'BCL2', 'HK2', 'p53', 'pSTAT3', 'MCL1', 'Caspase3',
       'ASCT2', 'LDHA', 'pErk', 'pP38', 'CD135', 'CD58', 'pS6', 'CPT1A',
       'CD36', 'MCT1', 'pCreb', 'CD68', 'cParp', 'MPO', 'GLUT1', 'CD11b'],
      dtype='object')

## Step 5 — Explore the annotated matrix

The matrix contains one row per cell. Metadata columns come first, followed by  
biological marker columns. Check the structure before saving.

In [10]:
# Preview
display(matrix.head())

# Summary statistics per plate
print("\nCells per plate:")
display(matrix.groupby('plate')['sample_name'].value_counts().rename('n_cells').reset_index())

# Cell type distribution
if 'celltypes' in matrix.columns:
    print("\nCell type distribution:")
    display(matrix['celltypes'].value_counts())

,original_filename,original_filename,Event_length,Time,dataset,panel,plate,sample_name,condition,celltypes,...,pS6,CPT1A,CD36,MCT1,pCreb,CD68,cParp,MPO,GLUT1,CD11b
3,Plate5_P16 Dx Untreated.csv,Plate5_P16 Dx Untreated.csv,17,439.09,OHSU_18,Panel1,Plate5,P16 Dx Untreated,Dx Untreated,89Y_CD45+/T-cells,...,16.9420,20.347,0.000,186.540,109.33,1.6480,2.95930,27.568,15.98800,0.12438
7,Plate5_P16 Dx Untreated.csv,Plate5_P16 Dx Untreated.csv,19,650.72,OHSU_18,Panel1,Plate5,P16 Dx Untreated,Dx Untreated,myeloid-enriched,...,3.0472,42.237,43.275,145.800,110.19,27.2170,2.13930,54.957,5.56670,82.75100
17,Plate5_P16 Dx Untreated.csv,Plate5_P16 Dx Untreated.csv,19,1730.70,OHSU_18,Panel1,Plate5,P16 Dx Untreated,Dx Untreated,megakaryocytic,...,9.3024,28.308,207.800,75.026,99.23,39.0700,6.63780,54.386,3.67610,9.62360
18,Plate5_P16 Dx Untreated.csv,Plate5_P16 Dx Untreated.csv,19,2108.20,OHSU_18,Panel1,Plate5,P16 Dx Untreated,Dx Untreated,myeloid-enriched,...,15.9320,27.387,36.786,138.190,114.92,58.8200,3.28800,37.453,0.49934,17.91800
20,Plate5_P16 Dx Untreated.csv,Plate5_P16 Dx Untreated.csv,17,2348.90,OHSU_18,Panel1,Plate5,P16 Dx Untreated,Dx Untreated,megakaryocytic,...,25.3090,33.155,221.340,273.420,120.26,6.6213,0.72709,17.588,1.35340,7.39170



Cells per plate:


,plate,sample_name,n_cells
0,Plate1,P2 EOT Untreated,155361
1,Plate1,P2 C2D15 Untreated,151139
2,Plate1,P2 C1D8 Untreated,149158
3,Plate1,P2 C3D1 Untreated,126789
4,Plate1,Healthy BM 38624,115847
...,...,...,...
106,Plate6,P6_1 Dx Rux,63941
107,Plate6,P6_1 Dx Ven,50358
108,Plate6,P6_1 Dx Combo,49573
109,Plate6,P6_1 Dx Untreated,36859



Cell type distribution:


celltypes
89Y_CD45+/T-cells    2411501
immature-like        1948534
megakaryocytic        879793
89Y_CD45+/B-cells     765315
myeloid-enriched      656138
Name: count, dtype: int64

## Step 6 — Save the matrix

Save the annotated matrix to disk so that `Correction.ipynb` can load it directly  
without re-processing the raw files.

In [ ]:
results_dir = "../results/"
os.makedirs(results_dir, exist_ok=True)

# Save the annotated matrix
matrix_path = os.path.join(results_dir, "matrix_OHSU.csv")
matrix.to_csv(matrix_path, index=False)
print(f"Matrix saved  : {matrix_path}")

# Save the marker list
markers_path = os.path.join(results_dir, "markers_OHSU.json")
with open(markers_path, "w") as f:
    json.dump(markers, f)
print(f"Markers saved : {markers_path}")

# ── Study config — single source of truth for all other notebooks ─────────────
study_dataset = matrix["dataset"].iloc[0] if "dataset" in matrix.columns else "unknown"
study_panel   = matrix["panel"].iloc[0]   if "panel"   in matrix.columns else "unknown"

matrix_corrected_path    = os.path.join(results_dir, "matrix_OHSU_corrected.csv")
matrix_intermediate_path = os.path.join(results_dir, "matrix_OHSU_intermediate.csv")

study_config = {
    "study_name":   study_dataset,
    "dataset":      study_dataset,
    "panel":        study_panel,
    "cytof_config": my_config,
    "outputs": {
        "matrix_path":              matrix_path,
        "markers_path":             markers_path,
        "matrix_corrected_path":    matrix_corrected_path,
        "matrix_intermediate_path": matrix_intermediate_path,
    },
}

study_config_path = os.path.join(results_dir, "study_config_OHSU.json")
with open(study_config_path, "w") as f:
    json.dump(study_config, f, indent=2)
print(f"Study config  : {study_config_path}")
print(f"\nIn Correction.ipynb set:")
print(f'  study_config_path = "{study_config_path}"')